# Limpeza e validação dos dados

O CSV de transações veio bem sujo (encoding, formatos misturados, duplicatas).
Esse notebook faz a limpeza e marca anomalias com Z-score.

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import sqlite3
import os
import re

os.makedirs("data/processed", exist_ok=True)
os.makedirs("outputs", exist_ok=True)

In [ ]:
df = pd.read_parquet("data/raw/transacoes.parquet")
df_ibov = pd.read_parquet("data/raw/ibovespa.parquet")
df_orc = pd.read_parquet("data/raw/orcamento.parquet")

print(f"transações: {len(df)}")
print(f"ibovespa:   {len(df_ibov)}")
print(f"orçamento:  {len(df_orc)}")

## Diagnóstico do CSV
Vamos ver o estrago...

In [ ]:
df.dtypes

In [ ]:
df.head(20)

In [ ]:
# quantidade de problemas
print("nulos por coluna:")
print((df == "").sum() + df.isnull().sum())
print(f"\nduplicatas: {df.duplicated().sum()}")

In [ ]:
# o valor tá como object pq tem "R$ 1.234,56" misturado
# vamos ver quantos registros tão nesse formato
mask_rs = df["valor"].astype(str).str.contains("R\$", na=False)
print(f"registros com 'R$': {mask_rs.sum()} de {len(df)}")

# exemplo:
df[mask_rs].head()

## Limpeza

In [ ]:
# 1. converter o campo valor
# tem que tratar os que vieram como "R$ 1.200,50"

def parse_valor(v):
    """tenta converter pra float, tratando formato brasileiro"""
    s = str(v).strip()
    if "R$" in s:
        s = s.replace("R$", "").strip()
        s = s.replace(".", "").replace(",", ".")
    try:
        return float(s)
    except ValueError:
        return np.nan

df["valor"] = df["valor"].apply(parse_valor)

nulos_valor = df["valor"].isna().sum()
print(f"valores que não consegui converter: {nulos_valor}")
print(f"dtype agora: {df['valor'].dtype}")

In [ ]:
# 2. padronizar datas
# tão em formatos misturados: yyyy-mm-dd, dd/mm/yyyy, dd-mm-yyyy...
# o pd.to_datetime com dayfirst=True resolve a maioria

# primeiro tento o jeito facil
df["data"] = pd.to_datetime(df["data"], dayfirst=True, errors="coerce")

print(f"datas que não converteram: {df['data'].isna().sum()}")
print(f"range: {df['data'].min()} a {df['data'].max()}")

In [ ]:
# 3. categorias - limpar espaço e padronizar casing
print("categorias encontradas (antes):")
print(df["categoria"].value_counts())

In [ ]:
# "alimentação" e "Alimentação" são a mesma coisa
# "Transporte " tem espaço no final...

df["categoria"] = df["categoria"].str.strip().str.title()

# substituir vazios por "Sem Categoria"
df.loc[df["categoria"].isin(["", "nan", "None"]) | df["categoria"].isna(), "categoria"] = "Sem Categoria"

print("categorias (depois):")
print(df["categoria"].value_counts())

In [ ]:
# 4. duplicatas
antes = len(df)
df = df.drop_duplicates()
print(f"removidas {antes - len(df)} duplicatas")

In [ ]:
# 5. dropar linhas sem data ou sem valor (inutilizáveis)
antes = len(df)
df = df.dropna(subset=["data", "valor"])
print(f"dropadas {antes - len(df)} linhas sem data/valor")

df = df.sort_values("data").reset_index(drop=True)
df.dtypes

In [ ]:
df.head()

## Detecção de anomalias - Z-score

Z-score > 3 (ou < -3) = o valor tá a mais de 3 desvios padrão da média.
Pode ser fraude, erro de digitação ou transação legítima mas atípica.

In [ ]:
z_scores = np.abs(stats.zscore(df["valor"]))
df["z_score"] = z_scores
df["anomalia"] = z_scores > 3

n_anom = df["anomalia"].sum()
print(f"anomalias: {n_anom} ({n_anom/len(df)*100:.1f}%)")
print(f"média: R$ {df['valor'].mean():,.2f}")
print(f"std:   R$ {df['valor'].std():,.2f}")

In [ ]:
# quais são as anomalias?
df[df["anomalia"]].sort_values("valor", ascending=False)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# histograma
ax1.hist(df["valor"], bins=60, color="steelblue", edgecolor="white", alpha=0.7)
limiar_pos = df["valor"].mean() + 3 * df["valor"].std()
limiar_neg = df["valor"].mean() - 3 * df["valor"].std()
ax1.axvline(limiar_pos, color="red", linestyle="--", label=f"3σ = {limiar_pos:,.0f}")
ax1.axvline(limiar_neg, color="red", linestyle="--", label=f"-3σ = {limiar_neg:,.0f}")
ax1.set_title("Distribuição dos valores")
ax1.set_xlabel("R$")
ax1.legend(fontsize=8)

# scatter
ok = df[~df["anomalia"]]
nok = df[df["anomalia"]]
ax2.scatter(ok["data"], ok["valor"], s=5, alpha=0.3, color="steelblue")
ax2.scatter(nok["data"], nok["valor"], s=35, color="red", zorder=5, label=f"{n_anom} anomalias")
ax2.set_title("Transações no tempo")
ax2.set_xlabel("Data")
ax2.set_ylabel("R$")
ax2.legend(fontsize=8)

plt.tight_layout()
plt.savefig("outputs/anomalias.png", dpi=150, bbox_inches="tight")
plt.show()

## Limpeza Ibovespa e orçamento

In [ ]:
# Ibovespa
if "Date" in df_ibov.columns:
    df_ibov = df_ibov.rename(columns={"Date": "data"})
df_ibov["data"] = pd.to_datetime(df_ibov["data"])

for col in ["Open", "High", "Low", "Close", "Volume"]:
    if col in df_ibov.columns:
        df_ibov[col] = pd.to_numeric(df_ibov[col], errors="coerce")

df_ibov = df_ibov.dropna(subset=["Close"]).sort_values("data").reset_index(drop=True)
df_ibov["retorno_diario"] = df_ibov["Close"].pct_change()

print(f"ibovespa: {len(df_ibov)} pregões")

In [ ]:
# Orçamento - tem várias categorias por mês, preciso agregar
df_orc["Mês"] = pd.to_datetime(df_orc["Mês"])

# somar a despesa planejada de todas as categorias pra ter o limite mensal
orc_mensal = df_orc.groupby("Mês").agg(
    Limite=("Despesa Planejada", "sum"),
    Receita_Planejada=("Receita Planejada", "first"),
).reset_index()

print(f"orçamento: {len(orc_mensal)} meses (agregado de {len(df_orc)} linhas)")
orc_mensal.head()

## Salvar

In [ ]:
df.to_parquet("data/processed/transacoes_clean.parquet", index=False)
df_ibov.to_parquet("data/processed/ibovespa_clean.parquet", index=False)
orc_mensal.to_parquet("data/processed/orcamento_clean.parquet", index=False)

print(f"transações: {len(df)} ({n_anom} anomalias marcadas)")
print(f"ibovespa:   {len(df_ibov)}")
print(f"orçamento:  {len(orc_mensal)}")
print("\ntudo em data/processed/")

## Salvar em SQLite

Jogar tudo num banco SQLite também — facilita fazer consultas ad-hoc
e fica mais fácil de compartilhar do que parquet.

In [ ]:
DB_PATH = "data/processed/financeiro.db"

conn = sqlite3.connect(DB_PATH)

df.to_sql("transacoes", conn, if_exists="replace", index=False)
df_ibov.to_sql("ibovespa", conn, if_exists="replace", index=False)
orc_mensal.to_sql("orcamento", conn, if_exists="replace", index=False)

# check rápido
for tabela in ["transacoes", "ibovespa", "orcamento"]:
    n = pd.read_sql(f"SELECT COUNT(*) as n FROM {tabela}", conn).iloc[0, 0]
    print(f"{tabela}: {n} linhas")

conn.close()
print(f"\nbanco salvo em {DB_PATH}")

In [ ]:
# teste rápido pra ver se os dados tão certos no banco
conn = sqlite3.connect(DB_PATH)

query = """
SELECT categoria,
       COUNT(*) as qtd,
       ROUND(AVG(valor), 2) as media,
       ROUND(SUM(valor), 2) as total
FROM transacoes
WHERE anomalia = 0
GROUP BY categoria
ORDER BY total DESC
LIMIT 10
"""
pd.read_sql(query, conn)